In [8]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

def interpolar_poblacion(archivo_csv, metodo='cubic'):
    """
    Interpola datos poblacionales quinquenales a anuales
    
    Parámetros:
    -----------
    archivo_csv : str
        Archivo CSV con columnas: Año, Alcaldía, Población total
    metodo : str
        'linear' o 'cubic'
    
    Retorna:
    --------
    DataFrame con población anual interpolada
    """
    
    # Cargar datos
    df = pd.read_csv(archivo_csv)
    df.columns = ['año', 'alcaldia', 'poblacion']  # Estandarizar nombres
    
    # Convertir población de string a número
    df['poblacion'] = df['poblacion'].astype(str).str.replace(',', '').astype(float)
    
    # Determinar rango completo de años
    año_min, año_max = df['año'].min(), 2024
    años_completos = list(range(año_min, año_max + 1))
    
    # Interpolar por alcaldía
    resultados = []
    
    for alcaldia in df['alcaldia'].unique():
        datos = df[df['alcaldia'] == alcaldia].sort_values('año')
        
        # Crear función de interpolación
        f = interp1d(datos['año'], datos['poblacion'], 
                    kind=metodo, fill_value='extrapolate')
        
        # Aplicar a todos los años
        poblacion_anual = f(años_completos)
        
        # Agregar a resultados
        for año, pob in zip(años_completos, poblacion_anual):
            resultados.append({
                'año': año,
                'alcaldia': alcaldia,
                'poblacion': max(0, int(pob))  # Evitar negativos
            })
    
    return pd.DataFrame(resultados)

def calcular_tasas_crecimiento(df):
    """
    Calcula tasas de crecimiento poblacional anuales
    """
    df = df.sort_values(['alcaldia', 'año'])
    df['tasa_crecimiento'] = df.groupby('alcaldia')['poblacion'].pct_change() * 100
    return df

# Uso del código
if __name__ == "__main__":
    # Interpolar datos
    df_anual = interpolar_poblacion('/home/adonnay_bazaldua/Documentos/GitHub/Nonparametric-analysis-of-homicides-in-CDMX-by-borough/data/processed/poblacion_total_tasa_crecimiento_1.1.csv')
    
    # Agregar tasas de crecimiento
    df_final = calcular_tasas_crecimiento(df_anual)
    
    # Guardar resultado
    df_final.to_csv('poblacion_anual_interpolada.csv', index=False)
    
    # Mostrar resumen
    print(f"Datos interpolados: {len(df_final)} observaciones")
    print(f"Alcaldías: {df_final['alcaldia'].nunique()}")
    print(f"Período: {df_final['año'].min()}-{df_final['año'].max()}")

Datos interpolados: 595 observaciones
Alcaldías: 17
Período: 1990-2024
